# __Project: Mandarin ASR with wav2vec 2.0 and CTC__

- This project follows the general workflow of the Hugging Face tutorial [Fine-Tuning XLS-R for Multilingual ASR with 🤗 Transformers](https://huggingface.co/blog/fine-tune-xlsr-wav2vec2). 

- The original tutorial has been adapted to the current Hugging Face Transformers API and to Mandarin speech recognition using the AISHELL-3 corpus.

- The goal of the project is to fine-tune a pretrained XLS-R model for Mandarin automatic speech recognition (ASR) using Connectionist Temporal Classification (CTC). Two output representations are investigated:

1. Chinese characters (Hanzi)
2. Tone-marked Pinyin syllables

- This notebook covers the third part of the project, including building data collator and fine-tuning the model.

- _AI-assisted development was used for code explanation, debugging, and API migration. Python version 3.13.14, Transformers version 5.12.1_

In [1]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch

from datasets import Audio, load_from_disk
from jiwer import cer, wer
from transformers import (
    Trainer,
    TrainingArguments,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForCTC,
    Wav2Vec2Processor,)

In [ ]:
PROJECT_ROOT = Path.cwd()

PROCESSED_DATA_DIR = (PROJECT_ROOT / "processed_data")

PROCESSOR_DIR = (PROJECT_ROOT / "processors")

MODEL_OUTPUT_DIR = (PROJECT_ROOT / "model_outputs")

In [3]:
dataset = load_from_disk(PROCESSED_DATA_DIR / "aishell3_dataset")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio_path', 'hanzi', 'pinyin', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
{'id': 'SSB03820023', 'speaker_id': 'SSB0382', 'audio_path': 'aishell3/train/wav/SSB0382/SSB03820023.wav', 'hanzi': '但这一点已经足够表明卷福的超高人气', 'pinyin': 'dan4 zhe4 yi4 dian2 yi3 jing1 zu2 gou4 biao3 ming2 juan4 fu2 de5 chao1 gao1 ren2 qi4', 'age_group': 'B', 'gender': 'female', 'accent': 'north'}


#### __📊 Data Collator__: Final preparation before fine-tuning

__What does the Data Collator contain?__
```
DataCollatorCTCWithPadding class
├── processor = hanzi_processor  → Hanzi collator
├── processor = pinyin_processor → Pinyin-syllable collator
└── processor = phone_processor  → Pinyin-character collator
```

__Why do we need Data Collator?__

- The data collator combines a list of individual examples into one batch. It dynamically pads the audio inputs and text labels separately, creates an attention mask for the padded audio, replaces label-padding positions with -100, and returns the tensors that the Trainer passes to the model.

__Basic structure of Data Collator?__
```
Dataset
   ↓
DataLoader selects several examples
   ↓
Data Collator creates a padded batch
   ↓
Trainer passes the batch to the model
   ↓
Model computes CTC loss
```

In [ ]:
PREPARED_DATA_DIR = PROCESSED_DATA_DIR / "prepared_ctc"

prepared_hanzi_dataset = load_from_disk(str(PREPARED_DATA_DIR / "hanzi"))
prepared_pinyin_dataset = load_from_disk(str(PREPARED_DATA_DIR / "pinyin"))
prepared_phone_dataset = load_from_disk(str(PREPARED_DATA_DIR / "phone"))

print(prepared_hanzi_dataset)

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor : Wav2Vec2Processor
    padding: bool | str = True

    '''
    Here the only args we give from the Trainer to the Data Collator is the feature of the input.

    features shaped like:
        features = [
            {
                "input_values": [...],
                "labels": [...],
            }]

    In the end, data collator will give back the batches in shape: 
    batch = {
        "input_values": torch.Tensor(...),
        "attention_mask": torch.Tensor(...),
        "labels": torch.Tensor(...),
    }

    One Example Input Contents:
    Input length: 78809
    Number of labels: 17
    true labels: 但这一点已经足够表明卷福的超高人气
    First input values: [ 0.04401884  0.02261921  0.00648973  0.03158857  0.03285171 -0.01215215
    -0.04094664 -0.03844285 -0.00987181  0.02702106]
    Labels: [151, 3512, 0, 2161, 1053, 2739, 3418, 775, 3199, 1600, 440, 2560, 2410, 3413, 3936, 93, 1879]

'''

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        # step 1: the data collator must get the audio input
        # input_values: the extracted features in single audiop file, e.g mel or mfcc
        input_features = [{"input_values": feature["input_values"]} for feature in features]

        # step 2: prepare the label data, then combine these two features together
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # step 3: padding the audio and the label
        # padding the audio using the feature extractor, audio padding using 0.0
        batch = (self.processor.feature_extractor.pad(input_features, padding = self.padding, return_tensors= "pt"))

        # padding labels using the tokenizer, label padding using -100
        labels_batch = (self.processor.tokenizer.pad(label_features, padding = self.padding, return_tensors = "pt"))

        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)

        batch["labels"] = labels

        return batch

In [ ]:
# creating data collator
hanzi_data_collator = (DataCollatorCTCWithPadding(processor = hanzi_processor, padding= True))

pinyin_data_collator = (DataCollatorCTCWithPadding(processor = pinyin_processor, padding= True))

phone_data_collator = (DataCollatorCTCWithPadding(processor = phone_processor, padding= True))

In [ ]:
# sanity check I: if the data collator works well. test with an example, show the shape of the batch and the labels
def check_batch(name, dataset, collator):
    train = dataset["train"]
    batch = collator([train[0], train[1]])

    print(f"\n{name} batch")
    for key in ("input_values", "attention_mask", "labels"):
        print(f"{key} shape:", batch[key].shape)
    print(batch["labels"])

    return batch


hanzi_batch = check_batch("Hanzi", prepared_hanzi_dataset, hanzi_data_collator)
pinyin_batch = check_batch("Pinyin", prepared_pinyin_dataset, pinyin_data_collator)
phone_batch = check_batch("Phone", prepared_phone_dataset, phone_data_collator)

In [ ]:
# sanity check II: assert the batch shape is correct, and the labels are correct
assert (hanzi_batch["input_values"].ndim == 2) # 2d shaped output, batch_size x audio_sequence_length
assert (hanzi_batch["attention_mask"].shape== hanzi_batch["input_values"].shape) # padding to the longest
assert hanzi_batch["labels"].ndim == 2 # 2d shaped output, batch_size x label_sequence_length

# check if the batch size is correct, and the labels are correct
assert (hanzi_batch["input_values"].shape[0]== 2)
assert (hanzi_batch["labels"].shape[0]== 2)

print("Hanzi data collator test passed.")

### __🏃‍♀️‍➡️Define the Trainer, and fine-tuning the XLR-wave2vec__
- We first fine-tune and validate the complete training process for the Hanzi recognition task.


In [ ]:
# define the task name and the corresponding 
# -----------------------------------------------
TASK = "hanzi"

tokenizer = hanzi_tokenizer
processor = hanzi_processor
prepared_dataset = prepared_hanzi_dataset
data_collator = hanzi_data_collator

# -----------------------------------------------
# TASK = "pinyin"

# tokenizer = pinyin_tokenizer
# processor = pinyin_processor
# prepared_dataset = prepared_pinyin_dataset
# data_collator = pinyin_data_collator

# -----------------------------------------------
# TASK = "phone"

# tokenizer = phone_tokenizer
# processor = phone_processor
# prepared_dataset = prepared_phone_dataset
# data_collator = phone_data_collator

print("Task:", TASK)
print("Vocab Size:", len(processor.tokenizer))
print(prepared_dataset)

#### __✔️ Define the Evaluation Metrics__
- for the Hanzi tokenization, we use Character Error Rate

- for the Pinyin tokenization, we use Word Error Rate

- for the Phone tokenization, we use the Edit Distance

Issues needs to be fixed: 
1. Phone edit distance needs to be normalized
2. Choose different metrics based on different metrics, but if we want to compare the model abilities variants under different settings, need to be normalized?
3. Levenshtein edit distance needs Chinese phone pair (sh to zh as a pair to be computed)
4. Pinyin WER actually is the syllable error rate

In [ ]:
# define the edit distance for the phone-level error
def edit_distance(reference, hypothesis):
    """compute levenshtein edit distance between two input sequences"""
    previous_row = list(range(len(hypothesis) + 1))

    for reference_index, reference_token in enumerate(reference, start=1):
        current_row = [reference_index]

        for hypothesis_index, hypothesis_token in enumerate(hypothesis, start=1):
            deletion_cost = (previous_row[hypothesis_index] + 1)
            
            insertion_cost = (current_row[hypothesis_index - 1] + 1)
            
            substitution_cost = (previous_row[hypothesis_index - 1]+ (reference_token != hypothesis_token))

            current_row.append(min(deletion_cost, insertion_cost, substitution_cost))
                            
        previous_row = current_row
            
    return previous_row[-1]

In [ ]:
# metrics for hanzi and pinyin are imported form the jiwer lib, we dont need to define them
def compute_metrics(prediction):
    prediction_logits = prediction.predictions

    if isinstance(prediction_logits, tuple):
        prediction_logits = prediction_logits[0]

    prediction_ids = np.argmax(prediction_logits, axis=-1)

    label_ids = prediction.label_ids.copy()

    label_ids[label_ids == -100] = (processor.tokenizer.pad_token_id)

    prediction_texts = processor.batch_decode(prediction_ids)

    reference_texts = processor.batch_decode(label_ids, group_tokens=False,)

    # Hanzi: Character Error Rate
    if TASK == "hanzi":
        return {
            "cer": cer(
                reference_texts,
                prediction_texts,
            )
        }

    # Pinyin syllable: Word/Syllable Error Rate
    if TASK == "pinyin":
        return {
            "wer": wer(
                reference_texts,
                prediction_texts,
            )
        }

    # Phone: Edit Distance
    if TASK == "phone":
        normalized_predictions = [
            "".join(text.lower().split())
            for text in prediction_texts
        ]

        normalized_references = [
            "".join(text.lower().split())
            for text in reference_texts
        ]

        distances = [edit_distance(list(reference), list(prediction)) for reference, prediction in zip(normalized_references, normalized_predictions)]

        total_distance = sum(distances)

        total_reference_length = sum(
            len(reference)
            for reference in normalized_references
        )

        return {
            "mean_edit_distance": float(
                np.mean(distances)
            ),
            "normalized_edit_distance": (
                total_distance
                / max(total_reference_length, 1)
            ),
        }

    raise ValueError(f"Unsupported task: {TASK}")

#### __➕ Laod the XLS-R Model__

In [ ]:
# laod the XLS-R model

CHECKPOINT = "facebook/wav2vec2-xls-r-300m"

model = Wav2Vec2ForCTC.from_pretrained(CHECKPOINT, attention_dropout = 0.0,
                                       hidden_dropout = 0.0, feat_proj_dropout = 0.0,
                                       mask_time_prob = 0.05, layerdrop = 0.0,
                                       ctc_loss_reduction = "mean", pad_token_id = processor.tokenizer.pad_token_id,
                                       vocab_size = len(processor.tokenizer),
                                       ignore_mismatched_sizes=True)

In [ ]:
# freeze the feature encoder to ensure the feature not updated during back prop.
model.freeze_feature_encoder()

In [ ]:
# sanity check: if it's fixed
frozen_parameters = sum(parameter.numel() for parameter in model.wav2vec2.feature_extractor.parameters() if not parameter.requires_grad)

print(frozen_parameters)

In [ ]:
smoke_train_dataset = (prepared_dataset["train"].select(range(8)))
smoke_eval_dataset = (prepared_dataset["validation"].select(range(4)))
print(smoke_train_dataset)
print(smoke_eval_dataset)

In [ ]:
import torch
from transformers import TrainingArguments

smoke_output_dir = (
    PROJECT_ROOT
    / "model_outputs"
    / TASK
    / "smoke_test"
)

training_args = TrainingArguments(
    output_dir=str(smoke_output_dir),

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,

    max_steps=2,
    learning_rate=3e-4,
    warmup_steps=0,

    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),

    eval_strategy="steps",
    eval_steps=1,

    save_strategy="no",
    logging_strategy="steps",
    logging_steps=1,

    report_to="none",
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=smoke_train_dataset,
    eval_dataset=smoke_eval_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

In [ ]:
smoke_result = trainer.train()
print(smoke_result)

smoke_metrics = trainer.evaluate()
print(smoke_metrics)

#### __✖️ Formal Trainer__

In [ ]:
print("CUDA avaliable:", torch.cuda.is_available) # check cuda
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
TASK = "hanzi"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "model_outputs"
    / TASK
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=3e-4,
    warmup_ratio=0.1,
    weight_decay=0.005,

    gradient_checkpointing=True,
    fp16=torch.cuda.is_available(),

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    save_total_limit=2,
    eval_accumulation_steps=10,

    dataloader_num_workers=4,
    dataloader_pin_memory=torch.cuda.is_available(),

    report_to="none",
    seed=42,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=hanzi_data_collator,
    train_dataset=prepared_hanzi_dataset["train"],
    eval_dataset=prepared_hanzi_dataset["validation"],
    processing_class=hanzi_processor,
    compute_metrics=compute_metrics,
)

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
FINAL_MODEL_DIR = OUTPUT_DIR / "final"

trainer.save_model(FINAL_MODEL_DIR)
hanzi_processor.save_pretrained(FINAL_MODEL_DIR)

trainer.save_state()

In [ ]:
test_metrics = trainer.evaluate(
    eval_dataset=prepared_hanzi_dataset["test"],
    metric_key_prefix="test",
)

print(test_metrics)

In [ ]:
trainer.log_metrics(
    "test",
    test_metrics,
)

trainer.save_metrics(
    "test",
    test_metrics,
)